In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="zeeshanparvez/andrew-v3", 
    repo_type="dataset", local_dir="./andrew-v3", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|██████████| 6/6 [00:02<00:00,  2.45it/s]


'/home/ubuntu/andrew-v3'

In [3]:
files = glob('andrew-v3/*/*.parquet')
len(files)

6

In [4]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [5]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 623/623 [01:57<00:00,  5.29it/s]


In [6]:
audio_files = [d['audio_filename'] for d in data]

with open('andrew-v3-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [9]:
# !zip -rq andrew-v3_audio.zip andrew-v3_audio
# !hf upload malaysia-ai/Multilingual-TTS andrew-v3_audio.zip --repo-type=dataset

In [2]:
# !zip -rq andrew-v3_audio_neucodec.zip andrew-v3_audio_neucodec
# !hf upload malaysia-ai/Multilingual-TTS andrew-v3_audio_neucodec.zip --repo-type=dataset

In [10]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'andrew-v3_audio/andrew-v3-data-train-00003-of-00006_0.mp3',
 'text': "like anxiety and depression. Now, I don't mean those in the medical term.",
 'speaker': 'andrew-v3_audio'}

In [11]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'andrew-v3')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 460.71ba/s]
Processing Files (1 / 1): 100%|██████████|  250kB /  250kB, 1.25MB/s  
New Data Upload: 100%|██████████|  250kB /  250kB, 1.25MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.65 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/01403cc323357787141cffc6753264f7c0f74d98', commit_message='Upload dataset', commit_description='', oid='01403cc323357787141cffc6753264f7c0f74d98', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)